###Deep Normalization exercise
In this exercise you will learn to use normaliztion and use your research skills in order to fine tune a CNN.


Talk with the tutor about the following:
1.   What is the defenition of normalization?
2.   Why should you use normalization?

**IMPORTANT NOTE: BEFORE IMPLEMENTATION READ WHAT THE NORMALIZATION IS AND DECIDE WHETHER YOU SHOULD TAKE SOMETHING FROM GIT OR IMPLEMENT IT YOURSELF**


~Don Shaked

In [ ]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torchvision
import numpy as np
import torch.optim as optim
import pickle
from torchvision import transforms
import random
from tqdm import tqdm

import os

import pandas as pd

drive_version = True
drive_cwd = "/content/drive/MyDrive/הכשרת מרכז המצוינות/metzuyanut מתקדם/Normalization"

if drive_version:
  from google.colab import drive
  drive.mount('/content/drive')

  os.chdir(drive_cwd)

zca_mat_path = "zca_mat.p"

results_path = "./results.csv"

print("CUDA is available: ", torch.cuda.is_available())

Mounted at /content/drive
CUDA is available:  True


In [ ]:
num_workers=0
batch_size=64

epochs = 2 #2
num_experiments = 5 #5
ratio_of_data_to_keep = 1 #1

results_df = pd.DataFrame(columns=["Name", "loss_train_per_batch", "acc_train", "loss_test_per_batch", "acc_test", "run_time_sec"])

Write the forward of the following CNN:

In [ ]:
class CnnVanilla(nn.Module):
  def __init__(self):
    super().__init__() # (N, 1, 28, 28)
    self.conv1 = nn.Conv2d(1, 6, 5) # (N, 6, 24, 24)
    self.pool_1 = nn.MaxPool2d(2, 2) # (N, 6, 12, 12)
    self.conv2 = nn.Conv2d(6, 16, 5) # (N, 16, 8, 8)
    self.pool_2 = nn.MaxPool2d(2, 2) # (N, 16, 4, 4)
    self.fc1 = nn.Linear(256, 120) # (N, 120)
    self.fc2 = nn.Linear(120, 84) # (N, 84)
    self.fc3 = nn.Linear(84, 10) # (N, 10)

    self.sequential = nn.Sequential(
      self.conv1,
      self.pool_1,
      nn.ReLU(),
      self.conv2,
      self.pool_2,
      nn.ReLU(),
      nn.Flatten(1),
      self.fc1,
      nn.ReLU(),
      self.fc2,
      nn.ReLU(),
      self.fc3,
    )
  def forward(self, x):
    return self.sequential(x)



In [ ]:
def download_mnist_data(curr_transforms, train_flag, ratio_to_keep=ratio_of_data_to_keep):
    dataset_train = torchvision.datasets.MNIST('../data', train=train_flag, download=True, transform=curr_transforms)
    train_keep_size = int(ratio_to_keep * len(dataset_train))
    subset = torch.utils.data.random_split(dataset_train, [train_keep_size, len(dataset_train) - train_keep_size])[0]
    return subset

In [ ]:
def reproduce(seed=42):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

In [ ]:

def save_results(df, path):

  if os.path.isfile(path):

    saved_df = pd.read_csv(path, index_col=0)[results_df.columns]

    new_df = pd.concat([df, saved_df], axis=0)
    new_df.drop_duplicates(subset="Name", inplace=True)

  else:
    new_df = df

  new_df.to_csv(path)

  return new_df

The follwoing is a sketch of the training process, you'll need to change it accordingly each time for the different methods you'll apply. You'll need to provide batch size, number of workers and learning rate. **Supply learning rate appropriate to batch size**

In [ ]:
def train_model(model, trainloader, testloader, epochs=2, optimizer=None, criterion=None, verbose=False, device=None):
    #===Logistics====

    if optimizer is None:
      momentum = 0.9
      lr = 1e-2
      optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)

    if device is None:
      device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
      model = model.to(device)

    if criterion is None:
      criterion = nn.CrossEntropyLoss()

    #===Train Loop===
    for epoch in range(epochs):
      running_loss = 0.
      for batch, (input, labels) in enumerate(trainloader):
        input, labels = input.to(device), labels.to(device)

        output = model(input)
        loss = criterion(output, labels)
        running_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if verbose:
          if batch%(len(trainloader)//5) == (len(trainloader)//5)-1:
            print(f"Epoch:{epoch+1}, Batch:{batch+1}, Training Loss:{running_loss/(batch+1):.4f}")

    #===Test Loop===
    with torch.no_grad():
      train_loss = 0.
      train_correct_predictions = 0
      for batch, (input, labels) in enumerate(trainloader):
        input, labels = input.to(device), labels.to(device)

        output = model(input)
        loss = criterion(output, labels)
        train_correct_predictions += torch.sum(torch.argmax(output,dim=-1) == labels).item()
        train_loss += loss.item()

      train_acc = train_correct_predictions / len(trainloader.dataset)
      # Loss for all of a signle batch
      train_loss = train_loss / len(trainloader)

      test_loss = 0.
      test_correct_predictions = 0
      for batch, (input, labels) in enumerate(testloader):
        input, labels = input.to(device), labels.to(device)

        output = model(input)
        loss = criterion(output, labels)
        test_correct_predictions += torch.sum(torch.argmax(output,dim=-1) == labels).item()
        test_loss += loss.item()

      test_acc = test_correct_predictions / len(testloader.dataset)
      test_loss = test_loss / len(testloader)

    return train_loss, train_acc, test_loss, test_acc

In [ ]:
import time

def run_experiment(model_class, trainloader, testloader, epochs=2, num_experiments=10, momentum = 0.9, lr = 1e-2, optimizer_class=None, criterion=None, verbose=True, device=None, seed=42):
  #===Logistics====

  reproduce(seed)

  if device is None:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

  if criterion is None:
    criterion = nn.CrossEntropyLoss()

  if optimizer_class is None:
    optimizer_class = optim.SGD

  start_time = time.time()
  #===Loops===

  # For all experiments
  loss_train_arr = []
  acc_train_arr = []
  loss_test_arr = []
  acc_test_arr = []
  for experiment in range(num_experiments):
    model = model_class()
    model = model.to(device)
    optimizer = optimizer_class(model.parameters(), lr=lr, momentum=momentum)
    train_loss, train_acc, test_loss, test_acc = train_model(model, trainloader, testloader, epochs, optimizer, criterion, verbose=True, device=device)

    loss_train_arr.append(train_loss)
    acc_train_arr.append(train_acc)
    loss_test_arr.append(test_loss)
    acc_test_arr.append(test_acc)

    if verbose:
      print(f"Experiment:{experiment}, train_loss:{train_loss:.2f}, train_acc:{train_acc*100:.2f}%, test_loss:{test_loss:.2f}, test_acc:{test_acc*100:.2f}%")

  # Time for a single experiment
  time_sec = (time.time() - start_time) / num_experiments

  loss_train = np.mean(loss_train_arr)
  acc_train = np.mean(acc_train_arr)
  loss_test = np.mean(loss_test_arr)
  acc_test = np.mean(acc_test_arr)

  return loss_train, acc_train, loss_test, acc_test, time_sec

In [ ]:
"""
def train_model(dataset_train, dataset_test, batch_size, num_workers, lr, net_builder, seed=42):
  losses = []
  accuracies_train = []
  accuracies_test = []
  total_pred_train = 60000
  batches = total_pred_train / batch_size
  epochs = 2
  reproduce(seed)

  criterion = nn.CrossEntropyLoss()
  train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=batch_size,
                                              shuffle=True, num_workers=num_workers)
  test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=batch_size,
                                              shuffle=True, num_workers=num_workers)
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  for _ in tqdm(range(10)):

        net = net_builder().to(device)
        optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9)
        for epoch in range(epochs):  # loop over the dataset multiple times
            for i, data in enumerate(train_loader, 0):
                # get the inputs; data is a list of [inputs, labels]
                inputs, labels = data
                inputs = inputs.to(device)
                labels = labels.to(device)
                # zero the parameter gradients
                optimizer.zero_grad()
                # forward + backward + optimize
                outputs = net(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                # print statistics
                if epoch == epochs-1:
                  running_loss += loss.item() / batches
        losses.append(running_loss)
        correct_pred = 0

        with torch.no_grad():
            for data in train_loader:
                inputs, labels = data
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = net(inputs)
                _, predictions = torch.max(outputs, 1)
                # collect the correct predictions for each class
                correct_pred += torch.sum(predictions == labels).cpu().numpy()
            accuracies_train.append(correct_pred/total_pred_train)
            for data in test_loader:
                inputs, labels = data
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = net(inputs)
                _, predictions = torch.max(outputs, 1)
                # collect the correct predictions for each class
                correct_pred += torch.sum(predictions == labels).cpu().numpy()
            accuracies_test.append(correct_pred/total_pred)
"""


"\ndef train_model(dataset_train, dataset_test, batch_size, num_workers, lr, net_builder, seed=42):\n  losses = []\n  accuracies_train = []\n  accuracies_test = []\n  total_pred_train = 60000\n  batches = total_pred_train / batch_size\n  epochs = 2\n  reproduce(seed)\n\n  criterion = nn.CrossEntropyLoss()\n  train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=batch_size,\n                                              shuffle=True, num_workers=num_workers)\n  test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=batch_size,\n                                              shuffle=True, num_workers=num_workers)\n  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n  for _ in tqdm(range(10)):\n\n        net = net_builder().to(device)\n        optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9)\n        for epoch in range(epochs):  # loop over the dataset multiple times\n            for i, data in enumerate(train_loader, 0):\n       

####Normalize Activation -  population based methods
Here we will investigate a lot of dfferent population normalization methods.

The methods to implement are:



1.   no normalizaiton - baseline
2.   centering - 1d
3.   centering - all d
4.   scaling - 1d
5.   scaling - all d
6.   standardizing - 1d
7.   standardizing - all d
8.   whitenning - ZCA matrix is saved as pickle and in the folder for usage.

Write the complexity of each method.

Which normalization will work the best on the dataset, write it here (before running the experiment):

Talk with the tutor why you thought this way.




#Answer:#

N = (Number of Samples)
D = (C*H*W)

1-7: O(N*D)

8: O(D^3 + D^2*N), Eigen decomposition + Matrix multiplication


Write each of the normaliztions as an augmentation, I provided the function of the whitening augmentation and provided the needed matrix in order for you to save time.

You'll need to calculate the statistics for each of the normalization methods.


In [ ]:
class Whitening(object):
    """Convert ndarrays in sample to Tensors."""

    def __init__(self, zca_mat_path):
      super().__init__()
      with open(zca_mat_path, 'rb') as file:
        self.zca_mat = pickle.load(file).type(torch.FloatTensor)


    def __call__(self, sample):
        f_sample = sample.flatten(1)
        proj_sample = torch.matmul(f_sample, self.zca_mat)

        return proj_sample.reshape((1, 28, 28))

class Normalize_1d(object):
    """Convert ndarrays in sample to Tensors."""

    def __init__(self, mean=None, std=None, eps=1e-2):
      super().__init__()
      # Assuming shape [H,W]
      self.mean = mean
      self.std = std
      self.eps = eps


    def __call__(self, sample):
        # Assuming sample shape [N, H, W]
        transformed_sample = sample

        if not (self.mean is None):
          transformed_sample = (transformed_sample - self.mean)

        if not (self.std is None):
          transformed_sample = transformed_sample / (self.std + self.eps)

        return transformed_sample


In [ ]:
# Dict of tuples (trainset, testset) with different normalizations
datasets = dict()

curr_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=(0.5,), std=(0.5,))])

dataset_train = download_mnist_data(curr_transform,True)
dataset_test = download_mnist_data(curr_transform,False)

datasets["Vanilla"] = (dataset_train, dataset_test)

# Calculating dataset statistics

curr_transform = transforms.Compose([transforms.ToTensor()])

dataset_train = download_mnist_data(curr_transform,True)
dataset_test = download_mnist_data(curr_transform,False)

dataset_train_data = np.array([data[0].numpy() for data in dataset_train])
dataset_test_data = np.array([data[0].numpy() for data in dataset_test])

mean_train = np.mean(dataset_train_data, axis=None)
mean_test = np.mean(dataset_test_data, axis=None)

std_train = np.std(dataset_train_data, axis=None)
std_test = np.std(dataset_test_data, axis=None)

mean_1d_train = np.mean(dataset_train_data, axis=0)
mean_1d_test = np.mean(dataset_test_data, axis=0)

std_1d_train = np.std(dataset_train_data, axis=0)
std_1d_test = np.std(dataset_test_data, axis=0)


# Creating datasets by order
curr_transform = transforms.Compose([transforms.ToTensor(), Normalize_1d(mean=mean_1d_train)])
dataset_train = download_mnist_data(curr_transform,True)
curr_transform = transforms.Compose([transforms.ToTensor(), Normalize_1d(mean=mean_1d_test)])
dataset_test = download_mnist_data(curr_transform,False)
datasets["Centering_1d"] = (dataset_train, dataset_test)

curr_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=(mean_train,), std=(1,))])
dataset_train = download_mnist_data(curr_transform,True)
curr_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=(mean_test,), std=(1,))])
dataset_test = download_mnist_data(curr_transform,False)
datasets["Centering_all_d"] = (dataset_train, dataset_test)

curr_transform = transforms.Compose([transforms.ToTensor(), Normalize_1d(std=std_1d_train)])
dataset_train = download_mnist_data(curr_transform,True)
curr_transform = transforms.Compose([transforms.ToTensor(), Normalize_1d(std=std_1d_test)])
dataset_test = download_mnist_data(curr_transform,False)
datasets["Scaling_1d"] = (dataset_train, dataset_test)

curr_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=(0,), std=(std_train,))])
dataset_train = download_mnist_data(curr_transform,True)
curr_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=(0,), std=(std_test,))])
dataset_test = download_mnist_data(curr_transform,False)
datasets["Scaling_all_d"] = (dataset_train, dataset_test)

curr_transform = transforms.Compose([transforms.ToTensor(), Normalize_1d(mean=mean_1d_train, std=std_1d_train)])
dataset_train = download_mnist_data(curr_transform,True)
curr_transform = transforms.Compose([transforms.ToTensor(), Normalize_1d(mean=mean_1d_test, std=std_1d_test)])
dataset_test = download_mnist_data(curr_transform,False)
datasets["Standardizing_1d"] = (dataset_train, dataset_test)

curr_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=(mean_train,), std=(std_train,))])
dataset_train = download_mnist_data(curr_transform,True)
curr_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=(mean_test,), std=(std_test,))])
dataset_test = download_mnist_data(curr_transform,False)
datasets["Standardizing_all_d"] = (dataset_train, dataset_test)


curr_transform = transforms.Compose([transforms.ToTensor(), Whitening(zca_mat_path)])
dataset_train = download_mnist_data(curr_transform,True)
dataset_test = download_mnist_data(curr_transform,False)
datasets["ZCA_Whitening"] = (dataset_train, dataset_test)


In [ ]:
# Run the expirement for all datasets
for dataset in datasets:
  print("Running for dataset: " + dataset)
  dataset_train, dataset_test = datasets[dataset]

  trainloader = DataLoader(dataset_train, batch_size=batch_size,
                                            shuffle=True, num_workers=num_workers)
  testloader = DataLoader(dataset_test, batch_size=batch_size,
                                              shuffle=False, num_workers=num_workers)
  expirement_vals = run_experiment(CnnVanilla, trainloader, testloader, epochs=epochs, num_experiments=num_experiments, criterion=nn.CrossEntropyLoss(), verbose=True, device=None)

  new_row_df = pd.DataFrame([[dataset] + list(expirement_vals)], columns=results_df.columns)
  results_df = pd.concat([results_df,new_row_df], ignore_index=True)
  save_results(results_df, results_path)


Running for dataset: Vanilla
Epoch:1, Batch:187, Training Loss:1.2592
Epoch:1, Batch:374, Training Loss:0.7322
Epoch:1, Batch:561, Training Loss:0.5298
Epoch:1, Batch:748, Training Loss:0.4254
Epoch:1, Batch:935, Training Loss:0.3595
Epoch:2, Batch:187, Training Loss:0.0726
Epoch:2, Batch:374, Training Loss:0.0729
Epoch:2, Batch:561, Training Loss:0.0736
Epoch:2, Batch:748, Training Loss:0.0723
Epoch:2, Batch:935, Training Loss:0.0708
Experiment:0, train_loss:0.04, train_acc:98.70%, test_loss:0.04, test_acc:98.59%
Epoch:1, Batch:187, Training Loss:1.5046
Epoch:1, Batch:374, Training Loss:0.8685
Epoch:1, Batch:561, Training Loss:0.6241
Epoch:1, Batch:748, Training Loss:0.4962
Epoch:1, Batch:935, Training Loss:0.4147
Epoch:2, Batch:187, Training Loss:0.0683
Epoch:2, Batch:374, Training Loss:0.0717
Epoch:2, Batch:561, Training Loss:0.0727
Epoch:2, Batch:748, Training Loss:0.0704
Epoch:2, Batch:935, Training Loss:0.0691
Experiment:1, train_loss:0.05, train_acc:98.53%, test_loss:0.06, test_

KeyboardInterrupt: ignored

Show both train accuracy and validation accuracy.

As the lecture says you will train using 2 epochs.
Write the results of each model to a CSV file that you'll show your tutor.


#### Normalize Activation - As function methods
Here we will investigate a lot of dfferent population normalization methods. On the convolution layers.... ADD

The methods to implement are:



1.   no normalizaiton - baseline
2.   batch norm - regular
3.   batch norm - standardizing all d
4.   layer norm
5.   group norm - 2 groups
6.   batch whitening - Use the one from PyTorch


write the the pros and cons of each each method. Talk with them with your tutor, about when you'll use each one.

Here you'll need to change the architecture of the network. Write each one as a different class with an appropriate name.



In [ ]:
# create a dic of models

class CnnNormalization(nn.Module):
  def __init__(self, normalization_module=None, conv2d_module_class=None):
    # Both normalization_module and conv2d_module_class should be len 2 array-like objects
    super().__init__()

    self.normalization_module = normalization_module
    self.conv2d_module_class = conv2d_module_class

    if self.normalization_module is None:
      self.normalization_module = [nn.Identity(), nn.Identity()]
    if self.conv2d_module_class is None:
      self.conv2d_module_class = [nn.Conv2d, nn.Conv2d]

    self.conv1 =self.conv2d_module_class[0](1, 6, 5)
    self.pool_1 = nn.MaxPool2d(2, 2) # (N, 6, 12, 12)
    self.conv2 = self.conv2d_module_class[1](6, 16, 5)
    self.pool_2 = nn.MaxPool2d(2, 2) # (N, 16, 4, 4)
    self.fc1 = nn.Linear(256, 120) # (N, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)


    self.sequential = nn.Sequential(
      self.conv1,
      self.normalization_module[0],
      self.pool_1,
      nn.ReLU(),
      self.conv2,
      self.normalization_module[1],
      self.pool_2,
      nn.ReLU(),
      nn.Flatten(1),
      self.fc1,
      nn.ReLU(),
      self.fc2,
      nn.ReLU(),
      self.fc3,
    )

  def forward(self, x):
    return self.sequential(x)

In [ ]:
import torch.nn
from torch.nn import Parameter


__all__ = ['pcaWhitening', 'PCAWhitening']


class PCAWhitening_Single(torch.nn.Module):
    def __init__(self, num_features, dim=4, eps=1e-3, momentum=0.1, affine=True,
                 *args, **kwargs):
        super(PCAWhitening_Single, self).__init__()
        # assert dim == 4, 'PCAWhitening is not support 2D'
        self.eps = eps
        self.momentum = momentum
        self.num_features = num_features
        self.affine = affine
        self.dim = dim
        shape = [1] * dim
        shape[1] = self.num_features

        self.register_buffer('running_mean', torch.zeros(num_features, 1))
        # running whiten matrix
        self.register_buffer('running_projection', torch.eye(num_features))


    def forward(self, X: torch.Tensor):
        x = X.transpose(0, 1).contiguous().view(self.num_features, -1)
        d, m = x.size()
        if self.training:
            # calculate centered activation by subtracted mini-batch mean
            mean = x.mean(-1, keepdim=True)
            self.running_mean = (1. - self.momentum) * self.running_mean + self.momentum * mean.data
            xc = x - mean
            # calculate covariance matrix
            sigma = torch.addmm(self.eps, torch.eye(self.num_features).to(X), 1. / m, xc, xc.transpose(0, 1))
            # reciprocal of trace of Sigma: shape [g, 1, 1]
            u, eig, _ = sigma.svd()
            scale = eig.rsqrt()
            wm = torch.matmul(scale.diag(),u.t())
            self.running_projection = (1. - self.momentum) * self.running_projection + self.momentum * wm.data
        else:
            xc = x - self.running_mean
            wm = self.running_projection
        xn = wm.mm(xc)
        Xn = xn.view(X.size(1), X.size(0), *X.size()[2:]).transpose(0, 1).contiguous()
        return Xn

class PCAWhitening(torch.nn.Module):
    def __init__(self, num_features, num_channels=16, dim=4, eps=1e-3, momentum=0.1, affine=True,
                 *args, **kwargs):
        super(PCAWhitening, self).__init__()
        # assert dim == 4, 'PCAWhitening is not support 2D'
        self.eps = eps
        self.momentum = momentum
        self.num_features = num_features
        self.num_channels = num_channels
        num_groups = (self.num_features-1) // self.num_channels + 1
        self.num_groups = num_groups
        self.PCAWhitening_Groups = torch.nn.ModuleList(
            [PCAWhitening_Single(num_features = self.num_channels, eps=eps, momentum=momentum) for _ in range(self.num_groups-1)]
        )
        num_channels_last=self.num_features - self.num_channels * (self.num_groups -1)
        self.PCAWhitening_Groups.append(PCAWhitening_Single(num_features = num_channels_last, eps=eps, momentum=momentum))

        print('PCAWhitening-------m_perGroup:' + str(self.num_channels) + '---nGroup:' + str(self.num_groups))
        self.affine = affine
        self.dim = dim
        shape = [1] * dim
        shape[1] = self.num_features
        if self.affine:
            self.weight = Parameter(torch.Tensor(*shape))
            self.bias = Parameter(torch.Tensor(*shape))
        else:
            self.register_parameter('weight', None)
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        # self.reset_running_stats()
        if self.affine:
            torch.nn.init.ones_(self.weight)
            torch.nn.init.zeros_(self.bias)

    def forward(self, X: torch.Tensor):
        X_splits = torch.split(X, self.num_channels, dim=1)
        X_hat_splits = []
        for i in range(self.num_groups):
            X_hat_tmp = self.PCAWhitening_Groups[i](X_splits[i])
            X_hat_splits.append(X_hat_tmp)
        X_hat = torch.cat(X_hat_splits, dim=1)
        # affine
        if self.affine:
            return X_hat * self.weight + self.bias
        else:
            return X_hat

    def extra_repr(self):
        return '{num_features}, num_channels={num_channels}, eps={eps}, ' \
               'momentum={momentum}, affine={affine}'.format(**self.__dict__)

In [ ]:
models_class = {}

# no normalizaiton - baseline

models_class["no_normalization"] = CnnNormalization

# batch norm - regular

models_class["batch_norm"]  = lambda: CnnNormalization(normalization_module=[nn.BatchNorm2d(6), nn.BatchNorm2d(16)])

# batch norm - standardizing all d

class BatchNorm2dAlld(nn.Module):
  def __init__(self):
    super().__init__()
    self.batch_norm = torch.nn.BatchNorm2d(1)

  def forward(self, input):
    shape = input.shape # (N,C,H,W)
    x = torch.reshape(input, (shape[0], 1, shape[2], -1))
    x = self.batch_norm(x)
    x = torch.reshape(input, shape)
    return x

models_class["batch_norm_all_d"] = lambda: CnnNormalization(normalization_module=[BatchNorm2dAlld(), BatchNorm2dAlld()])

# layer norm

models_class["batch_norm"]  = lambda: CnnNormalization(normalization_module=[nn.LayerNorm([6, 24, 24]), nn.LayerNorm([16, 8, 8])])

# group norm - 2 groups

models_class["group_norm"]  = lambda: CnnNormalization(normalization_module=[nn.GroupNorm(2,6), nn.GroupNorm(2,16)])

# batch whitening - Use the one from PyTorch

models_class["batch_whitening"]  = lambda: CnnNormalization(normalization_module=[PCAWhitening(6), PCAWhitening(16)])

#### Normalize weights - As function methods
Here we will investigate 3 weight normalization methods.

The methods to implement are:



1.  Normalization Propagation
2.  Centered Weight Normalization
3.  Orthogonal Weight Normalization


What's the motivation behind each method ?
Here you'll need to change both architecture and the learning procedure



In [ ]:

class NormalizationPropagation2d(nn.Module):
  def __init__(self, in_channels):
    super().__init__()
    self.bn = nn.BatchNorm2d(in_channels)

  def forward(self, input):
    x = self.bn(input)
    x = nn.functional.relu(x)
    x = (x - np.sqrt(1/(2*np.pi))) / (0.5 * (1 - 1/np.pi))
    return x

# NEW
class NormalizationPropagation2dGroupNorm(nn.Module):
  def __init__(self, group_size, in_channels):
    super().__init__()
    self.bn = nn.GroupNorm(group_size, in_channels)

  def forward(self, input):
    x = self.bn(input)
    x = nn.functional.relu(x)
    x = (x - np.sqrt(1/(2*np.pi))) / (0.5 * (1 - 1/np.pi))
    return x

class CenteredWeightNormConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True):
        super(CenteredWeightNormConv2d, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.weight_g = nn.Parameter(torch.ones(out_channels))
        self.bias = None
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(self, x):
        weight = self.conv.weight
        mean = weight.mean(dim=[1, 2, 3], keepdim=True)
        centered_weight = weight - mean
        std = centered_weight.view(centered_weight.size(0), -1).std(dim=1, keepdim=True).view(-1, 1, 1, 1)
        normalized_weight = centered_weight / std
        normalized_weight = normalized_weight * self.weight_g.view(-1, 1, 1, 1)

        if self.bias is not None:
            return self.conv(x) + self.bias.view(1, -1, 1, 1)
        else:
            return self.conv(x)

class Conv2DWithOWN(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(Conv2DWithOWN, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding)
        self.own()

    def forward(self, x):
        x = self.conv(x)
        return x

    def own(self):
        weight = self.conv.weight.data
        _, _, kh, kw = weight.size()
        weight_reshaped = weight.view(self.conv.out_channels, -1)
        u, _, v = torch.svd(weight_reshaped)
        weight_orthogonal = torch.matmul(u, v.t()).view(weight.size())
        self.conv.weight.data.copy_(weight_orthogonal)

In [ ]:
# Normalization Propagation

models_class["normalization_propagration"]  = lambda: CnnNormalization(normalization_module=[NormalizationPropagation2d(6), NormalizationPropagation2d(16)])

# Centered Weight Normalization

models_class["centered_weight_normalization"] = lambda: CnnNormalization(conv2d_module_class=[CenteredWeightNormConv2d, CenteredWeightNormConv2d])

# Orthogonal Weight Normalization

models_class["orthogonal_weight_normalization"] = lambda: CnnNormalization(conv2d_module_class=[Conv2DWithOWN, Conv2DWithOWN])

In [ ]:
# Run the expirement for all datasets
for model_class in models_class:
  print("Running for Normalization Activation: " + model_class)
  dataset_train, dataset_test = datasets["Vanilla"]

  trainloader = DataLoader(dataset_train, batch_size=batch_size,
                                            shuffle=True, num_workers=num_workers)
  testloader = DataLoader(dataset_test, batch_size=batch_size,
                                              shuffle=False, num_workers=num_workers)
  expirement_vals = run_experiment(models_class[model_class], trainloader, testloader, epochs=epochs, num_experiments=num_experiments, criterion=nn.CrossEntropyLoss(), verbose=True, device=None)

  new_row_df = pd.DataFrame([[model_class] + list(expirement_vals)], columns=results_df.columns)
  results_df = pd.concat([results_df,new_row_df], ignore_index=True)
  save_results(results_df, results_path)

#### Normalize Gradients - As function methods
Here we will investigate 2 gradients normalization methods.

The methods to implement are:



1.  Gradient centralization
2.  Layer-wise Adaptive Rate Scaling (LARS)


What's the motivation behind each method ?
Here you'll need to the learning procedure



In [ ]:
# dic of optimizers

class LARS(torch.optim.Optimizer):
  def __init__(self, params, total_number_of_steps, lr=0.01, momentum=0.99, weight_decay=0.0001, eta=0.01):
    defaults = dict(lr=lr, momentum=momentum, weight_decay=weight_decay, eta=eta,total_number_of_steps=total_number_of_steps)
    super().__init__(params, defaults)

  def step(self):

    for group in self.param_groups:
      momentum = group["momentum"]
      weight_decay = group["weight_decay"]
      lr = group["lr"]
      eta = group["eta"]
      T = group["total_number_of_steps"]

      for p in group["params"]:
        if p.grad is None:
          continue

        param_state = self.state[p]

        if len(param_state) == 0:
          param_state["step"] = 0
          param_state["momentum_state"] = torch.zeros_like(p).to(p.device)

        step_num = param_state["step"]
        momentum_state = param_state["momentum_state"]

        weight_norm = torch.norm(p.data)
        grad_norm = torch.norm(p.grad.data)

        global_lr = lr * (1 - (step_num+1) / T)
        local_lr = eta * weight_norm / (grad_norm + weight_decay * weight_norm)

        momentum_state.data.mul_(momentum)
        momentum_state.data.add_(global_lr*local_lr*(p.grad.data + weight_decay * p.data))

        p.data.add_(-momentum_state.data)

        param_state["step"] += 1


class GradientCentralization(optim.Optimizer):
    def __init__(self, params, lr=1e-3, **kwargs):
        defaults = dict(lr=lr)
        super(GradientCentralization, self).__init__(params, defaults)

    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            for param in group['params']:
                if param.grad is None:
                    continue

                grad = param.grad.data
                if grad.is_sparse:
                    raise RuntimeError("Gradient Centralization doesn't support sparse gradients")

                if grad.dim() > 1:
                    grad.add_(-grad.mean())

                param.data.add_(grad, alpha=-group['lr'])

        return loss


In [ ]:
optimizers = {}

optimizers["LARS"] = lambda params, **args: LARS(params=params, total_number_of_steps=epochs*int(len(datasets["Vanilla"][0])/batch_size), **args)

optimizers["gradient_centralization"] = GradientCentralization

In [ ]:
# Run the expirement for all datasets
for optimizer in optimizers:
  print("Running for Optimizer: " + optimizer)
  dataset_train, dataset_test = datasets["Vanilla"]

  trainloader = DataLoader(dataset_train, batch_size=batch_size,
                                            shuffle=True, num_workers=num_workers)
  testloader = DataLoader(dataset_test, batch_size=batch_size,
                                              shuffle=False, num_workers=num_workers)

  expirement_vals = run_experiment(models_class["no_normalization"], trainloader, testloader, optimizer_class=optimizers[optimizer], epochs=epochs, num_experiments=num_experiments, criterion=nn.CrossEntropyLoss(), verbose=True, device=None)

  new_row_df = pd.DataFrame([[optimizer] + list(expirement_vals)], columns=results_df.columns)
  results_df = pd.concat([results_df,new_row_df], ignore_index=True)
  save_results(results_df, results_path)

In [ ]:
# print the table

results_df = save_results(results_df, results_path)

display(results_df)


#### Curse of Dimensionality

1. Here you'll combine the best methods from each section, and try to find other methods permutation that will be better. Combine the best results from each section.

2. After that try to find a better permutation.

3. In projects there are so many possible ways to solve the problem, so many permutation available.
What are the true possible permutation space of this problem?


In [ ]:
# Best combination results:

# Standarizing all d
# group norm - 2 groups
# Normalization Propagation
# Normal optimizer SGD

# can't do both normalization propagration and group norm
# doing a mix

#models_class["group_norm"]  = lambda: CnnNormalization(normalization_module=[nn.GroupNorm(2,6), nn.GroupNorm(2,16)])
#models_class["normalization_propagration"]  = lambda: CnnNormalization(normalization_module=[NormalizationPropagation2d(6), NormalizationPropagation2d(16)])

name = "Standardizing_all_d_normalization_propagration_group_norm__SGD"
models_class[name]  = lambda: CnnNormalization(normalization_module=[NormalizationPropagation2d(6), nn.GroupNorm(2,16)])

print("Started training best combination")
dataset_train, dataset_test = datasets["Standardizing_all_d"]

trainloader = DataLoader(dataset_train, batch_size=batch_size,
                                          shuffle=True, num_workers=num_workers)
testloader = DataLoader(dataset_test, batch_size=batch_size,
                                            shuffle=False, num_workers=num_workers)

expirement_vals = run_experiment(models_class[name], trainloader, testloader, epochs=epochs, num_experiments=num_experiments, criterion=nn.CrossEntropyLoss(), verbose=True, device=None)

new_row_df = pd.DataFrame([[name] + list(expirement_vals)], columns=results_df.columns)
results_df = pd.concat([results_df,new_row_df], ignore_index=True)
save_results(results_df, results_path)


In [ ]:
# Nornmalization propataion with group norm inside

name = "Standardizing_all_d_normalization_propagration_group_norm_inside_SGD"
models_class[name]  = lambda: CnnNormalization(normalization_module=[NormalizationPropagation2dGroupNorm(2,6), NormalizationPropagation2dGroupNorm(2,16)])

print("Started training best combination")
dataset_train, dataset_test = datasets["Standardizing_all_d"]

trainloader = DataLoader(dataset_train, batch_size=batch_size,
                                          shuffle=True, num_workers=num_workers)
testloader = DataLoader(dataset_test, batch_size=batch_size,
                                            shuffle=False, num_workers=num_workers)

expirement_vals = run_experiment(models_class[name], trainloader, testloader, epochs=epochs, num_experiments=num_experiments, criterion=nn.CrossEntropyLoss(), verbose=True, device=None)

new_row_df = pd.DataFrame([[name] + list(expirement_vals)], columns=results_df.columns)
results_df = pd.concat([results_df,new_row_df], ignore_index=True)
save_results(results_df, results_path)

Started training best combination
Epoch:1, Batch:187, Training Loss:0.5082
Epoch:1, Batch:374, Training Loss:0.3236
Epoch:1, Batch:561, Training Loss:0.2487
Epoch:1, Batch:748, Training Loss:0.2077
Epoch:1, Batch:935, Training Loss:0.1802
Epoch:2, Batch:187, Training Loss:0.0675
Epoch:2, Batch:374, Training Loss:0.0622
Epoch:2, Batch:561, Training Loss:0.0620
Epoch:2, Batch:748, Training Loss:0.0587
Epoch:2, Batch:935, Training Loss:0.0580
Experiment:0, train_loss:0.05, train_acc:98.46%, test_loss:0.05, test_acc:98.48%
Epoch:1, Batch:187, Training Loss:0.5006
Epoch:1, Batch:374, Training Loss:0.3183
Epoch:1, Batch:561, Training Loss:0.2525
Epoch:1, Batch:748, Training Loss:0.2094
Epoch:1, Batch:935, Training Loss:0.1832
Epoch:2, Batch:187, Training Loss:0.0682
Epoch:2, Batch:374, Training Loss:0.0635
Epoch:2, Batch:561, Training Loss:0.0615
Epoch:2, Batch:748, Training Loss:0.0594
Epoch:2, Batch:935, Training Loss:0.0580
Experiment:1, train_loss:0.04, train_acc:98.85%, test_loss:0.04, 

,Name,loss_train_per_batch,acc_train,loss_test_per_batch,acc_test,run_time_sec
0,Vanilla,0.054665,0.983053,0.056324,0.98148,62.173076
1,Standardizing_all_d_normalization_propagration...,0.044656,0.985690,0.050034,0.98394,54.386486
0,Standardizing_all_d_normalization_propagration...,0.050684,0.983893,0.054112,0.98268,57.854168
1,Standardizing_all_d_group_norm_SGD,0.051077,0.983770,0.052246,0.98284,51.256520
0,LARS,0.497978,0.849917,0.478183,0.85582,57.380851
1,gradient_centralization,0.226135,0.929553,0.210226,0.93408,51.033737
2,no_normalization,0.052586,0.983620,0.052697,0.98260,52.576394
3,batch_norm,0.051409,0.984003,0.051712,0.98314,53.694029
4,batch_norm_all_d,0.051925,0.983723,0.052259,0.98306,51.669353
5,group_norm,0.050797,0.984073,0.050529,0.98406,51.259005


In [ ]:
# My combination:

# Standarizing all d
# group norm - 2 groups
# Normal optimizer SGD

name = "Standardizing_all_d_group_norm_SGD"
models_class[name]  = lambda: CnnNormalization(normalization_module=[nn.GroupNorm(2,6), nn.GroupNorm(2,16)])

print("Started training my combination")
dataset_train, dataset_test = datasets["Standardizing_all_d"]

trainloader = DataLoader(dataset_train, batch_size=batch_size,
                                          shuffle=True, num_workers=num_workers)
testloader = DataLoader(dataset_test, batch_size=batch_size,
                                            shuffle=False, num_workers=num_workers)

expirement_vals = run_experiment(models_class[name], trainloader, testloader, epochs=epochs, num_experiments=num_experiments, criterion=nn.CrossEntropyLoss(), verbose=True, device=None)

new_row_df = pd.DataFrame([[name] + list(expirement_vals)], columns=results_df.columns)
results_df = pd.concat([results_df,new_row_df], ignore_index=True)
save_results(results_df, results_path)


In [ ]:
# (Theoretical permutation space) = 8*7*4*3

# (True possible permutation space) = 8*7*4*3 - 1
# normalization propagation as a layer normalization method

#### Final Words
You've learned a lot of different normalization methods in this exercise. If you want to review the material you've learned here you can watch the lecture by me "Normalization over 9000" it's in the Mador Lectures.

Hope you learned a lot,
Don Shaked